In [ ]:
# ==============================================================================
# Cellule 1 : Installations et Importations nécessaires
# ==============================================================================
!pip install -q einops transformers datasets

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer

print("PyTorch version:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

In [ ]:
# ==============================================================================
# Cellule 2 : Implémentation du mécanisme Single-Head Attention
# ==============================================================================
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        
        # Projections linéaires pour Query, Key, et Value
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        
        # Variable pour stocker les poids d'attention afin de les inspecter/visualiser
        self.attention_weights = None

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        b, s, d = x.shape
        
        # 1. Calcul des projections Q, K, V
        Q = self.q_linear(x)  # (b, s, d)
        K = self.k_linear(x)  # (b, s, d)
        V = self.v_linear(x)  # (b, s, d)
        
        # 2. Scaled Dot-Product Attention
        # Score = (Q * K^T) / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d) # (b, s, s)
        
        # 3. Softmax pour obtenir la distribution de probabilité
        self.attention_weights = F.softmax(scores, dim=-1) # (b, s, s)
        
        # 4. Multiplier par la matrice Value
        output = torch.matmul(self.attention_weights, V) # (b, s, d)
        
        return output

# --- Validation des formes (Shapes) ---
batch_size, seq_len, hidden_dim = 2, 8, 64
dummy_input = torch.randn(batch_size, seq_len, hidden_dim)

sha = SingleHeadAttention(d_model=hidden_dim)
output_sha = sha(dummy_input)

print("=== Validation Single-Head Attention ===")
print("Shape d'entrée :", dummy_input.shape)
print("Shape de sortie :", output_sha.shape)
print("Shape des poids d'attention mémorisés :", sha.attention_weights.shape)

In [ ]:
# ==============================================================================
# Cellule 3 : Implémentation du mécanisme Multi-Head Attention
# ==============================================================================
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model doit être divisible par num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Projections initiales
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)
        
        # Projection de sortie finale
        self.out_linear = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.attention_maps = None  # Pour enregistrer les cartes d'attention des têtes

    def forward(self, x):
        b, s, d = x.shape
        
        # 1. Projections et reshape pour séparer les têtes
        # (b, s, d) -> (b, s, num_heads, d_k) -> (b, num_heads, s, d_k)
        Q = self.q_linear(x).view(b, s, self.num_heads, self.d_k).transpose(1, 2)
        K = self.k_linear(x).view(b, s, self.num_heads, self.d_k).transpose(1, 2)
        V = self.v_linear(x).view(b, s, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. Scaled Dot-Product sur l'ensemble des têtes en parallèle
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k) # (b, num_heads, s, s)
        
        attn_weights = F.softmax(scores, dim=-1)
        self.attention_maps = attn_weights  # Sauvegarde pour l'analyse
        
        attn_weights = self.dropout(attn_weights)
        
        # 3. Aggégration avec les valeurs
        context = torch.matmul(attn_weights, V) # (b, num_heads, s, d_k)
        
        # 4. Concatenation des têtes
        # (b, num_heads, s, d_k) -> (b, s, num_heads, d_k) -> (b, s, d)
        context = context.transpose(1, 2).contiguous().view(b, s, d)
        
        # 5. Projection de sortie linéaire finale
        return self.out_linear(context)

# --- Validation des formes (Shapes) ---
num_heads = 8
mha = MultiHeadAttention(d_model=hidden_dim, num_heads=num_heads)
output_mha = mha(dummy_input)

print("=== Validation Multi-Head Attention ===")
print("Shape d'entrée :", dummy_input.shape)
print("Shape de sortie :", output_mha.shape)
print("Shape de la carte d'attention multi-têtes :", mha.attention_maps.shape)

In [ ]:
# ==============================================================================
# Cellule 4 : Custom Transformer Encoder Block Stacks
# ==============================================================================
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Bloc Feed-Forward (MLP)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # Attention + Connexion Résiduelle + Layer Normalization
        attn_out = self.mha(x)
        x = self.norm1(x + attn_out)
        
        # Feed-Forward + Connexion Résiduelle + Layer Normalization
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

# Test de notre couche d'encodage complète
encoder_layer = TransformerEncoderLayer(d_model=hidden_dim, num_heads=num_heads)
output_layer = encoder_layer(dummy_input)
print("Shape après passage dans l'Encoder Layer personnalisé :", output_layer.shape)

In [ ]:
# ==============================================================================
# Cellule 5 : Visualisation des cartes d'attention (Attention Maps)
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# On récupère l'attention calculée sur notre échantillon factice (Batch 0, Tête 0)
sample_attn_map = mha.attention_maps[0, 0].detach().cpu().numpy()

plt.figure(figsize=(6, 5))
sns.heatmap(sample_attn_map, annot=True, cmap="viridis", fmt=".2f")
plt.title("Carte d'attention (Batch 0, Head 0)")
plt.xlabel("Tokens Clés (Keys)")
plt.ylabel("Tokens Requêtes (Queries)")
plt.show()

In [ ]:
# ==============================================================================
# Cellule 6 : Réflexion Analytique & Comparaison Méthodologique (Rapport écrit)
# ==============================================================================
print("=== COMPTE RENDU TECHNIQUE : COMPARAISON DES ARCHITECTURES D'ATTENTION ===")
print("-" * 75)

report = """
1. Single-Head vs. Multi-Head Attention:
   L'attention à tête unique force le modèle à converger vers une seule distribution d'importance globale sur l'axe textuel. À l'inverse, l'attention multi-têtes (Multi-Head Attention) fragmente l'espace vectoriel imbriqué d_model en sous-espaces d_k indépendants. Cela donne la flexibilité au modèle de suivre simultanément des dynamiques distinctes (par exemple, des dépendances purement grammaticales d'un côté et des résolutions sémantiques ou contextuelles de l'autre).

2. Avantages d'un Stack Encodeur Custom léger:
   - Coût d'apprentissage et de calcul faible.
   - Moins sujet au surapprentissage si le volume de données local est réduit.
   - Idéal pour une implémentation embarquée (Edge AI) ou sur architecture restreinte.

3. Avantages des modèles pré-entraînés (BERT / DistilBERT):
   - Comprennent nativement la syntaxe complexe grâce à un pré-entraînement massif auto-supervisé.
   - Nécessitent extrêmement peu de données labellisées lors du fine-tuning pour atteindre un score optimal.
   - Robustesse face aux anomalies de saisie, synonymes complexes ou formulations croisées.
"""
print(report)